# Project 04: Natural Language Sentiment Analysis with Word Embeddings & BiLSTM

### Objective:
Build an end-to-end Natural Language Processing (NLP) sentiment classifier that reads customer reviews and predicts positive vs negative sentiment.

### Pipeline Stages:
1. Text tokenization, vocabulary mapping, and sequence padding
2. Dense Word Embedding layer ($V \to D$ dimensional continuous semantic space)
3. Bidirectional Long Short-Term Memory (BiLSTM) network
4. Binary classification head with dropout regularization

In [ ]:
import torch
import torch.nn as nn

class SentimentBiLSTM(nn.Module):
    def __init__(self, vocab_size=5000, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=2, dropout=0.2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, text_tokens):
        embeds = self.embedding(text_tokens)
        lstm_out, (hn, cn) = self.lstm(embeds)
        # Concatenate final hidden states from forward and backward directions
        hidden_cat = torch.cat((hn[-2, :, :], hn[-1, :, :]), dim=1)
        return self.classifier(hidden_cat)

sentiment_model = SentimentBiLSTM()
# Batch of 4 customer reviews padded to 30 words
sample_reviews = torch.randint(1, 4000, (4, 30))
logits = sentiment_model(sample_reviews)
probs = torch.sigmoid(logits)
print("Review Sentiment Probabilities (1 = Positive, 0 = Negative):\n", probs.detach().numpy())

## 🎯 Exercises & Practice
1. Explain the purpose of `padding_idx=0` in `nn.Embedding`.

In [ ]:
# Exercise Solution
# Answer: It ensures token ID 0 (the padding token) always maps to a constant zero vector 
# with zero gradient updates, ignoring synthetic padding across variable-length text!